### Задание 1
Напишите код для подсчёта `BLEU` при `n=2`. Используйте формулу, приведённую выше.

Чтобы выполнить задание, используйте библиотеку `evalaute`. Установите её с помощью пакетного менеджера `pip`

In [30]:
import math
from collections import Counter

def compute_bleu(candidate, reference, max_order=2):
    cand_tokens = candidate.split()
    ref_tokens = reference.split()
    
    precisions = []
    for n in range(1, max_order+1):
        # === ВАШ КОД ===
        # Посчитайте n-граммную точность для разных n
        cand_ngrams = Counter([tuple(cand_tokens[i:i+n]) for i in range(len(cand_tokens) - n + 1)])
        ref_ngrams = Counter([tuple(ref_tokens[i:i+n]) for i in range(len(ref_tokens) - n + 1)])
        
        overlap = {k: min(v, ref_ngrams[k]) for k, v in cand_ngrams.items()}
        p_n = sum(overlap.values()) / sum(cand_ngrams.values())
        precisions.append(p_n)

    # Если хотя бы одна precision == 0, BLEU = 0
    if any(p == 0 for p in precisions):
        return 0.0
    
    # Brevity Penalty
    c, r = len(cand_tokens), len(ref_tokens)
    BP = 1 if c > r else math.exp(1-r/c)
    
    bleu = BP * math.exp(sum([math.log(p) for p in precisions])/len(precisions))
    return bleu

candidate = "Ходор долго держал дверь"
reference = "Ходор долго закрыл дверь"

print("BLEU (ваша реализация): ", compute_bleu(candidate, reference))

import evaluate
reference_bleu = evaluate.load("bleu")
results = reference_bleu.compute(predictions=[candidate], references=[reference], tokenizer=lambda x: x.split(), max_order=2)
print("BLEU (референс): ", results["bleu"])

BLEU (ваша реализация):  0.49999999999999994
BLEU (референс):  0.49999999999999994


### Задание 2
Напишите код для подсчёта ROUGE-L. Верните точность, полноту и F1-меру.

In [41]:
def longest_common_sequence(X, Y):  # поиск lcs с помощью динамического программирования
    m, n = len(X), len(Y)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m):
        for j in range(n):
            if X[i] == Y[j]:
                dp[i+1][j+1] = dp[i][j] + 1
            else:
                dp[i+1][j+1] = max(dp[i][j+1], dp[i+1][j])
    return dp[m][n]

def rouge_l(candidate, reference):
    # === ВАШ КОД ===
    cand_tokens, ref_tokens = candidate.split(), reference.split()
    lcs = longest_common_sequence(cand_tokens, ref_tokens)
    precision = lcs / len(cand_tokens)
    recall = lcs / len(ref_tokens)
    f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1

candidate, reference = "Ходор держал дверь", "Ходор держал дверь, чтобы Бран мог спастись"
print("Ваш Rouge-L: ", rouge_l(candidate, reference)[-1])

reference_rouge = evaluate.load('rouge')

print("Референсный Rouge-L: ", reference_rouge.compute(predictions=[candidate], references=[reference], tokenizer=lambda x: x.split())['rougeL'])

Ваш Rouge-L:  0.4
Референсный Rouge-L:  0.4


### Задание 3
Реализуйте BERTScore без поправки на idf. За пример возьмите расчёт полноты.

In [45]:
import torch
from transformers import AutoTokenizer, AutoModel

# Загружаем модель
tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")
model = AutoModel.from_pretrained("cointegrated/rubert-tiny2")
model.eval()

def get_embeddings(text: str):
    """Возвращает эмбеддинги токенов без CLS/SEP."""
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)
    with torch.no_grad():
        outputs = model(**inputs)
    # Последний слой: [batch, seq_len, hidden_size]
    embeddings = outputs.last_hidden_state.squeeze(0)
    return embeddings

def bertscore_pair(hyp, ref):
    # Получаем эмбеддинги
    h = get_embeddings(hyp)   # [len_h, d]
    r = get_embeddings(ref)   # [len_r, d]

    # Нормировка эмбеддингов
    # == ВАШ КОД ==
    h = torch.nn.functional.normalize(h, p=2, dim=1)
    r = torch.nn.functional.normalize(r, p=2, dim=1)

    # Косинусное расстояние
    # == ВАШ КОД
    sim = torch.matmul(h, r.T)

    # Precision: для каждого токена h берем max по r
    P = sim.max(dim=1).values.mean().item()
    # Recall: для каждого токена r берем max по h
    R = sim.max(dim=0).values.mean().item()
    # F1
    F1 = 2 * P * R / (P + R)

    return P, R, F1

# Пример
hyp = "Сегодня будет краткий дождь и прохладный ветер."
ref = "Сегодня ожидается непродолжительный дождь и прохладный ветер."

P, R, F1 = bertscore_pair(hyp, ref)
print(f"P={P:.4f}, R={R:.4f}, F1={F1:.4f}") # P=0.9045, R=0.9048, F1=0.9047

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 5774.82it/s]
[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


P=0.9045, R=0.9048, F1=0.9047
